# Wildfire Web Inference Pipeline Preparation
สมุดงานนี้ปรับปรุงตามแนวทางเพื่อนำไปใช้งานบน Web Backend โดยแบ่งออกเป็น 2 ส่วนหลัก:
1. **One-Time Setup:** สร้างและบันทึกข้อมูล `Baseline` (จากอดีต) และ `KMeans Model` สำหรับนำไปเตรียมไว้บน Server/Database
2. **Web Backend Pipeline:** ฟังก์ชัน `preprocess_for_inference` สำหรับรับข้อมูลสภาวะอากาศ/ดาวเทียมของ "1 เดือนล่าสุด" มาแปลงเป็นฟีเจอร์สำหรับโมเดล CatBoost

In [9]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# สร้างโฟลเดอร์สำหรับเก็บไฟล์ Assets ของเว็บ
os.makedirs('../Web_Assets', exist_ok=True)

## Part 1: One-Time Setup สำหรับเตรียม Database และโมเดล

In [10]:
print("--- 1. เตรียม Baseline Data ---")
# โหลดข้อมูลประวัติทั้งหมด
df_raw = pd.read_csv('../Dataset/df_final_model.csv')

# คำนวณ Baseline (ค่าเฉลี่ยรายเดือนของแต่ละจังหวัดในอดีต) เพื่อใช้หา Anomaly
baseline_df = df_raw.groupby(['NAME_1', 'month'])[['ndvi', 'soil_moisture', 'temp']].mean().reset_index()
baseline_df.columns = ['NAME_1', 'month', 'ndvi_base', 'moisture_base', 'temp_base']

# บันทึก Baseline ลง CSV สำหรับนำไป Import เข้า Database ของเว็บ
baseline_df.to_csv('../Web_Assets/baseline_table.csv', index=False)
print("✅ Saved: Web_Assets/baseline_table.csv (ตารางนี้เอาไปใส่ DB ได้เลย)")

--- 1. เตรียม Baseline Data ---
✅ Saved: Web_Assets/baseline_table.csv (ตารางนี้เอาไปใส่ DB ได้เลย)


In [11]:
print("--- 2. เทรนและบันทึกโมเดล KMeans (Cluster ID) ---")
# ดึงข้อมูลมาหาค่าเฉลี่ยรายอำเภอก่อนนำไปเทรน KMeans (เพื่อให้คลัสเตอร์เหมือนกับตอนเทรนโมเดลหลัก)
num_cols = ['ndvi', 'soil_moisture', 'temp', 'elev', 'slope']
df_train_cluster = df_raw.groupby(['NAME_1', 'NAME_2', 'month'])[num_cols].mean().reset_index()

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df_train_cluster[num_cols].fillna(0))

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_cluster)

# บันทึก Scaler และ KMeans Model ให้เว็บโหลดไปใช้งาน
joblib.dump(scaler, '../Web_Assets/cluster_scaler.pkl')
joblib.dump(kmeans, '../Web_Assets/cluster_kmeans.pkl')
print("✅ Saved: ../Web_Assets/cluster_scaler.pkl และ ../Web_Assets/cluster_kmeans.pkl")

--- 2. เทรนและบันทึกโมเดล KMeans (Cluster ID) ---
✅ Saved: ../Web_Assets/cluster_scaler.pkl และ ../Web_Assets/cluster_kmeans.pkl


## Part 2: Web Backend Pipeline (ฟังก์ชันสำหรับแปลงข้อมูลในเว็บ)

In [19]:
import numpy as np
import pandas as pd

def preprocess_for_inference(df_current, baseline_df, scaler, kmeans, feature_list_path='../Models/model_d_features.csv'):
    """
    ฟังก์ชันสำหรับ Web Backend:
    1. รับข้อมูลเดือนล่าสุด (df_current)
    2. รับตาราง Baseline (ค่าเฉลี่ยจาก Train เท่านั้น) เพื่อคำนวณ Anomaly
    3. คำนวณ Physics & Cyclical Features
    4. ทำ Clustering
    5. คืนค่าเฉพาะคอลัมน์ที่โมเดลต้องการ
    """
    df = df_current.copy()
    
    # --- Group A: Physics-based Feature Engineering ---
    # คำนวณตามสูตรเดียวกับตอนเทรน
    df['veg_stress'] = (df['swir1'] - df['nir']) / (df['swir1'] + df['nir'] + 1e-6)
    df['fire_weather_idx'] = df['temp'] * (1.0 - df['soil_moisture'].clip(0, 1))
    df['wind_speed'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)
    df['drought_proxy'] = (1 - df['ndvi'].clip(-1, 1)) * (1 - df['soil_moisture'].clip(0, 1))
    df['terrain_roughness'] = df['slope'] * np.log1p(df['elev'])
    df['hot_dry_stress'] = df['temp'] * df['veg_stress']

    # --- Group B: Historical Anomaly (ใช้ Baseline ที่เตรียมไว้ล่วงหน้า) ---
    # baseline_df ต้องประกอบด้วย ['NAME_1', 'month', 'ndvi_base', 'moisture_base', 'temp_base']
    df = df.merge(baseline_df, on=['NAME_1', 'month'], how='left')
    
    df['ndvi_anomaly'] = (df['ndvi'] - df['ndvi_base']).fillna(0)
    df['moisture_anomaly'] = (df['soil_moisture'] - df['moisture_base']).fillna(0)
    df['temp_anomaly'] = (df['temp'] - df['temp_base']).fillna(0)
    
    # ลบคอลัมน์ baseline ออกเพื่อให้ DataFrame สะอาดเหมือนตอนเทรน
    cols_to_drop = ['ndvi_base', 'moisture_base', 'temp_base']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # --- Group C: Month Encoding (Cyclical) ---
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # --- Group D: Unsupervised Features (Clustering) ---
    try:
        # scaler.feature_names_in_ คือรายชื่อคอลัมน์ตามลำดับที่ใช้ตอน fit
        cluster_feats = scaler.feature_names_in_.tolist()
    except AttributeError:
        # กรณี scaler รุ่นเก่าที่ไม่มี feature_names_in_ ให้ระบุเองให้ตรงกับตอนเทรน
        # สมมติว่าเป็น: ['elev', 'ndvi', 'soil_moisture', 'slope', 'temp']
        cluster_feats = ['elev', 'ndvi', 'soil_moisture', 'slope', 'temp'] 
    
    # ตรวจสอบว่าใน df มีคอลัมน์เหล่านี้ครบไหม
    X_cluster = df[cluster_feats].fillna(0)
    
    # ตอนนี้ลำดับคอลัมน์ใน X_cluster จะตรงกับที่ scaler ต้องการแล้ว
    X_scaled = scaler.transform(X_cluster)
    df['cluster_id'] = kmeans.predict(X_scaled).astype(str)

    # --- Group E: Column Alignment ---
    # ดึงรายชื่อฟีเจอร์ที่บันทึกไว้ตอนเทรน เพื่อเรียงลำดับคอลัมน์ให้ถูกต้อง
    try:
        expected_cols = pd.read_csv(feature_list_path)['feature'].tolist()
        
        # กรณีมีฟีเจอร์ใหม่ที่ใน df_current ไม่มี (เช่น ลืมใส่ NAME_1 หรือคอลัมน์อื่นๆ)
        for col in expected_cols:
            if col not in df.columns:
                df[col] = 0  # เติม 0 สำหรับฟีเจอร์ที่ขาด (กันพัง)
                
        return df[expected_cols]
    except Exception as e:
        print(f"⚠️ Warning: Could not align features from CSV: {e}")
        return df

## Part 3: จำลองการทำงาน (Testing Web Inference)

In [20]:
# --------------------------------------------------------
# ในระบบจริง โค้ดส่วนนี้จะทำงานเมื่อกด "ประเมินความเสี่ยง" บนเว็บ
# --------------------------------------------------------

# 1. โหลดโมเดล/ข้อมูล (บนเว็บอาจจะโหลดไว้แล้วใน Memory)
loaded_baseline = pd.read_csv('../Web_Assets/baseline_table.csv')
loaded_scaler = joblib.load('../Web_Assets/cluster_scaler.pkl')
loaded_kmeans = joblib.load('../Web_Assets/cluster_kmeans.pkl')

# 2. รับข้อมูลภาพถ่ายดาวเทียมและสภาพอากาศ "1 เดือนล่าสุด" (จำลองโดยดึงเฉพาะเดือนเมษายนจากข้อมูลดิบ)
base_cols = ['ndvi', 'ndwi', 'nbr', 'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 
             'temp', 'soil_moisture', 'wind_u', 'wind_v', 'elev', 'slope', 'aspect', 
             'month', 'NAME_1', 'NAME_2']

mock_current_data = df_raw[df_raw['month'] == 4][base_cols].groupby(['NAME_1', 'NAME_2', 'month']).mean(numeric_only=True).reset_index()
# สมมติค่า landcover (ในความจริงข้อมูลดาวเทียมจะมี)
mock_current_data['landcover'] = '1' 

print("📥 1. ข้อมูลเข้าสู่เว็บ (Input Data Shape):", mock_current_data.shape)

# 3. ประมวลผลผ่าน Pipeline
final_features = preprocess_for_inference(
    df_current=mock_current_data, 
    baseline_df=loaded_baseline, 
    scaler=loaded_scaler, 
    kmeans=loaded_kmeans
)

print("✅ 2. ฟีเจอร์เตรียมเสร็จสิ้น (Output Features Shape):", final_features.shape)
print("พร้อมส่งเข้า CatBoost โมเดล! \n")
final_features.head()

📥 1. ข้อมูลเข้าสู่เว็บ (Input Data Shape): (178, 20)
✅ 2. ฟีเจอร์เตรียมเสร็จสิ้น (Output Features Shape): (178, 32)
พร้อมส่งเข้า CatBoost โมเดล! 



,ndvi,ndwi,nbr,blue,green,red,nir,swir1,swir2,temp,...,wind_speed,drought_proxy,terrain_roughness,hot_dry_stress,ndvi_anomaly,moisture_anomaly,temp_anomaly,month_sin,month_cos,cluster_id
0,0.394185,-0.374679,0.266893,1445.500000,1587.500000,1521.500000,3490.750000,2935.250000,2041.000000,26.645518,...,1.055165,0.422181,47.632302,-2.303390,0.005113,0.001256,0.171056,0.866025,-0.5,0
1,0.378845,-0.385069,0.250666,1500.000000,1771.000000,1797.000000,3989.000000,3586.000000,2390.000000,26.132350,...,0.692523,0.435212,77.579932,-1.390276,-0.010226,-0.002513,-0.342112,0.866025,-0.5,0
2,0.328317,-0.326750,0.162095,1604.333333,1835.870370,1831.753086,3632.080247,3630.370370,2627.925926,27.953835,...,1.056166,0.524539,51.645352,-0.006581,-0.031132,-0.052054,0.268024,0.866025,-0.5,0
3,0.348701,-0.339223,0.192035,1568.568293,1794.104878,1759.919512,3653.863415,3451.285366,2496.704878,27.794370,...,1.054663,0.503411,84.118899,-0.792458,-0.010748,-0.044056,0.108559,0.866025,-0.5,0
4,0.341918,-0.348720,0.165048,1527.157609,1777.010870,1808.722826,3724.793478,3667.076087,2675.663043,28.411096,...,1.205912,0.473739,77.510694,-0.221840,-0.017531,0.009000,0.725285,0.866025,-0.5,0
